In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ipynb

### Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

In [ ]:
!pip install regex
from TextPreprocessing import text_cleaning_pipeline

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [ ]:
import regex as re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

def lower_order(text):
  return text.lower()

def remove_urls(text):
  # Remove URLs
  text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
  # Remove mentions (@username)
  text = re.sub(r'@\w+', '', text)
  return text

def remove_emoji(text):
  # Remove emojis
  text = re.sub(r'\p{Emoji_Modifier_Base}\p{Emoji_Modifier}*', '', text, flags=re.UNICODE)
  text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]', '', text, flags=re.UNICODE)
  return text

def removeunwanted_characters(text):
  # Remove punctuation and numbers
  text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
  text = re.sub(r'\d+', '', text)
  # Remove all other unwanted characters (non-alphanumeric, extra spaces)
  text = re.sub(r'[^a-zA-Z\s]', '', text) # Keep only letters and spaces
  text = re.sub(r'\s+', ' ', text).strip() # Replace multiple spaces with single space
  return text

### Build a Text Cleaning Pipeline

In [ ]:
def text_cleaning_pipeline(dataset, rule = "lemmatize"):

  # Convert the input to small/lower order.
  data = lower_order(dataset)

  # Remove URLs
  data = remove_urls(data)

  # Remove emojis
  data = remove_emoji(data)

  # Remove all other unwanted characters.
  data = removeunwanted_characters(data)

  # Create tokens.
  tokens = data.split()

  wordnet = WordNetLemmatizer()
  porter = PorterStemmer()
  stop_words = set(stopwords.words('english'))
  stop_words.update(['@', 'RT', 'rt'])

  tokens = [word for word in tokens if word not in stop_words]

  if rule == "lemmatize":
    tokens = [wordnet.lemmatize(token, pos='v') for token in tokens]
  elif rule == "stem":
    tokens = [porter.stem(token) for token in tokens]
  else:
    print("Pick between lemmatize or stem")

  return " ".join(tokens)

### Load the Dataset

In [ ]:
import pandas as pd

dataset_path = '/content/drive/MyDrive/6S012 Artificial Intelligence/Workshop/Week-8/trum_tweet_sentiment_analysis.csv'
df_sentiment = pd.read_csv(dataset_path)

display(df_sentiment.head())
print(df_sentiment.info())

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1850123 entries, 0 to 1850122
Data columns (total 2 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   text       object
 1   Sentiment  int64 
dtypes: int64(1), object(1)
memory usage: 28.2+ MB
None


### Text Cleaning and Tokenization

In [ ]:
df_sentiment['cleaned_text'] = df_sentiment['text'].apply(text_cleaning_pipeline)

print("DataFrame after applying text cleaning pipeline:")
display(df_sentiment.head())

DataFrame after applying text cleaning pipeline:


,text,Sentiment,cleaned_text
0,RT @JohnLeguizamo: #trump not draining swamp b...,0,trump drain swamp taxpayer dollars trip advert...
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0,icymi hackers rig fm radio station play antitr...
2,Trump protests: LGBTQ rally in New York https:...,1,trump protest lgbtq rally new york bbcworld via
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0,hi im piers morgan david beckham awful donald ...
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0,tech firm sue buzzfeed publish unverified trum...


### Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df_sentiment['cleaned_text']
y = df_sentiment['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")
print("Train-test split completed.")

Training set size: 1480098 samples
Testing set size: 370025 samples
Train-test split completed.


### TF-IDF Vectorization
TF-IDF (Term Frequency-Inverse Document Frequency) is a numerical statistic that reflects how important a word is to a document in a collection or corpus.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Fit the vectorizer on the training data and transform both training and testing data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")
print("TF-IDF Vectorization completed.")

Shape of X_train_tfidf: (1480098, 5000)
Shape of X_test_tfidf: (370025, 5000)
TF-IDF Vectorization completed.


### Model Training and Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_tfidf, y_train)
print("Model training completed.")

y_pred = logistic_model.predict(X_test_tfidf)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model training completed.

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95    248563
           1       0.91      0.87      0.89    121462

    accuracy                           0.93    370025
   macro avg       0.92      0.91      0.92    370025
weighted avg       0.93      0.93      0.93    370025



## **Precision**

**Class 0 (Negative Sentiment):** The model is correct `94%` of the time when it preditcs a tweet. There are few false positives for negative sentiment.

**Class 1 (Positive Sentiment):** When the model predicts a tweet is positive, it is correct `91%` of the time.


## **Recall**

**Class 0 (Negative Sentiment):** The model correctly identifies `96%` of all actual negative tweets. The model misses very few true negative tweets (few false negatives).

**Class 1 (Positive Sentiment):** The model correctly identifies `87%` of all actual positive tweets. This indicates there are more false negatives for positive sentiment compared to negative sentiment.

## **F1-Score**

**Class 0 (Negative Sentiment):** The F1-score is the harmonic mean of precision and recall. There seems to be a good balance between precision and recall for negative sentiment as indicated by the score of `0.95`.

**Class 1 (Positive Sentiment):** An F1-score of `0.89` for positive sentiment is also strong.

## **Support**

**Class 0 (Negative Sentiment):** The number of actual negative tweets in the test set is `248,563`

**Class 1 (Positive Sentiment):** The number of actual positive tweets in your test set is `121,462`

## **Accuracy**

**Overall Accuracy:** `93%` of all predictions made by the model were correct.

## **Macro Avg and Weighted Avg**

**Macro Avg (Precision, Recall, F1-Score):** The macro average calculates the metric independently for each class and then takes the average. It treats all classes equally. The scores (around `0.92`) indicate strong performance across both classes.

**Weighted Avg (Precision, Recall, F1-Score):** The weighted average accounts for the number of instances in each class. Given the slight imbalance (more negative tweets), this average provides a more realistic view of the model's overall performance, aligning closely with the overall accuracy (around `0.93`).

### Feature Importance: Top Words for Each Sentiment

In [ ]:
import pandas as pd

# feature names from the TF-IDF vectorizer
feature_names = tfidf_vectorizer.get_feature_names_out()

# the coefficients from the Logistic Regression model
coefficients = logistic_model.coef_[0]

feature_importance = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})

feature_importance = feature_importance.sort_values(by='coefficient', ascending=False)

print("\nTop 10 words for Positive Sentiment:")
display(feature_importance.head(10))

print("\nTop 10 words for Negative Sentiment:")
display(feature_importance.tail(10))


Top 10 words for Positive Sentiment:


,feature,coefficient
2935,new,17.307838
3297,please,15.003341
3827,sanctuary,14.739876
663,care,14.032358
4280,super,13.546199
4593,truth,13.088993
4855,well,13.077750
1898,good,12.861285
485,bless,12.743239
236,approval,12.664183



Top 10 words for Negative Sentiment:


,feature,coefficient
1472,enemy,-11.034714
2858,murder,-11.076960
2772,mislead,-11.083232
4763,violate,-11.262420
3518,racist,-11.396497
3517,racism,-11.592036
1828,funeral,-11.802043
1037,criticism,-12.393502
1817,fuck,-12.743180
3847,scandal,-13.459092


These are the words that the model relies on most heavily. We can see terms strongly associated with 'Positive' (Sentiment 1) and 'Negative' (Sentiment 0) classifications.

### Error Analysis: Displaying Misclassified Samples

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    'text': X_test,
    'actual_sentiment': y_test,
    'predicted_sentiment': y_pred
})

misclassified_df = results_df[results_df['actual_sentiment'] != results_df['predicted_sentiment']]# Identify misclassified samples

print(f"Total misclassified samples: {len(misclassified_df)}")
display(misclassified_df.head(10))

Total misclassified samples: 26981


,text,actual_sentiment,predicted_sentiment
1713952,merrily tweet morning hadnt see president trum...,1,0
428846,people within fortresseurope oppose trump plan...,1,0
1529409,new flynn resignation raise question trump adm...,0,1
746604,trump scotus pick isnt merrill garland need st...,0,1
1640876,people connect trump campaign administration e...,1,0
225173,expect trumpism shortly say cancel wasnt ask come,1,0
1756515,bbcworld bbcbreaking donald trump choice labou...,1,0
188194,environmentalists worry trump supreme court no...,0,1
1640141,top story trump campaign aid repeat contact ru...,0,1
396249,notable casualness trump vow destroy someone h...,1,0
